1 Count property types
Load the file. Use value_counts() on property_type and show only the five most common
types. What share of all listings is Entire rental unit? (Use normalize=True, rounded to
three decimals.)

Answer: ~53%

In [60]:
import pandas as pd

listings = pd.read_csv('listings.csv')
listings['property_type'].value_counts().head(5)
listings['property_type'].value_counts(normalize=True).round(3)

property_type
Entire rental unit                    0.534
Entire home                           0.124
Private room in rental unit           0.107
Entire condo                          0.074
Entire villa                          0.033
Entire townhouse                      0.016
Room in hotel                         0.014
Private room in condo                 0.011
Private room in home                  0.011
Entire loft                           0.009
Entire serviced apartment             0.009
Private room in bed and breakfast     0.007
Private room in hostel                0.007
Entire guesthouse                     0.006
Tiny home                             0.005
Private room in villa                 0.004
Entire cabin                          0.003
Entire guest suite                    0.003
Room in aparthotel                    0.003
Room in boutique hotel                0.003
Shared room in hotel                  0.002
Boat                                  0.002
Private room in to

2 The smallest neighbourhoods
Use value_counts() on neighbourhood_cleansed. Which three neighbourhoods have the
fewest listings, and how many do they have?

Answer: 
Rinkeby-Tensta    108
Skärholmens        96
Spånga-Tensta      89

In [61]:
listings['neighbourhood_cleansed'].value_counts().tail(3)

neighbourhood_cleansed
Rinkeby-Tensta    108
Skärholmens        96
Spånga-Tensta      89
Name: count, dtype: int64

3 Average beds per room type
Calculate the average of beds for each room_type, rounded to two decimals. Then use
agg(['count', 'mean']) on the same thing. Which average do you trust the least, and
why?

Answer: 1.00 for hotel rooms, as it is based on 3 listings only

In [62]:
listings.groupby('room_type')['beds'].agg(['count', 'mean']).round(2)

,count,mean
room_type,,
Entire home/apt,3680,2.48
Hotel room,3,1.00
Private room,766,1.48
Shared room,24,4.58


4 Reviews per neighbourhood
Group by neighbourhood_cleansed and calculate count, mean and median of
number_of_reviews, rounded to two decimals, sorted by the median. Which neighbourhood
has the lowest median and which the highest? For Södermalms the mean is much bigger
than the median. What could be the reason?

Answer:
lowest median - Skärholmens
highest median - Södermalms
The big diff between mean and median can be caused by several properties,
which have much more reviews than others.

In [63]:
(listings
 .groupby('neighbourhood_cleansed')['number_of_reviews']
 .agg(['count', 'mean', 'median'])
 .round(2)
 .sort_values('median')
)

,count,mean,median
neighbourhood_cleansed,,,
Skärholmens,96,21.29,3.0
Farsta,177,18.18,4.0
Hässelby-Vällingby,148,14.94,4.0
Rinkeby-Tensta,108,16.44,5.0
Bromma,281,19.16,5.0
Enskede-Årsta-Vantörs,445,22.55,7.0
Spånga-Tensta,89,17.01,7.0
Hägersten-Liljeholmens,464,22.92,8.0
Skarpnäcks,221,22.55,8.0


5 Your own summary table
Make a summary per room_type with named columns: n_listings (count of id), avg_beds
(mean of beds) and max_reviews (max of number_of_reviews). Round to two decimals.


In [64]:
(listings
 .groupby('room_type')
 .agg(n_listings=('id', 'count'), 
      avg_beds=('beds', 'mean'), 
      max_reviews=('number_of_reviews', 'max'))
 .round(2)
)
 

,n_listings,avg_beds,max_reviews
room_type,,,
Entire home/apt,3760,2.48,1293
Hotel room,3,1.00,79
Private room,787,1.48,867
Shared room,24,4.58,510


6 Two categories at once
Make a crosstab of neighbourhood_cleansed and host_is_superhost with the share in
percent for each neighbourhood (normalize='index', times 100, one decimal). The column
t means that the host is a superhost. Which neighbourhood has the highest share of
superhosts, and which the lowest?

Answer:
highest share - Södermalms (23.8)
lowest share - Bromma (7.1)

In [65]:
((pd.crosstab(listings['neighbourhood_cleansed'], 
              listings['host_is_superhost'], 
              normalize='index') * 100.0)
 .round(1)
 .sort_values('t', ascending=False)
)

host_is_superhost,f,t
neighbourhood_cleansed,,
Södermalms,76.2,23.8
Spånga-Tensta,77.5,22.5
Älvsjö,80.4,19.6
Norrmalms,81.0,19.0
Östermalms,83.2,16.8
Skärholmens,84.4,15.6
Skarpnäcks,86.9,13.1
Rinkeby-Tensta,87.0,13.0
Hägersten-Liljeholmens,87.5,12.5


7 Create new columns
(a) Create a column has_reviews that is True when number_of_reviews is bigger than 0.
How many listings are False?
(b) Create a column popularity with np.where: 'popular' if the listing has 50 reviews or
more, otherwise 'other'. Count the values.

In [66]:
import numpy as np

listings['has_reviews'] = listings['number_of_reviews'] > 0
print((listings['has_reviews'] == False).sum()) # 644

listings['popularity'] = np.where(listings['number_of_reviews'] >= 50, 'popular', 'other')
listings['popularity'].value_counts() # other: 3805, popular: 769

644


popularity
other      3805
popular     769
Name: count, dtype: int64

8 Turn numbers into groups
The column minimum_nights says how long you have to stay. Make four groups with pd.cut:
1 night, 2-6 nights, 7-29 nights and 30 or more nights. Save them as a column stay with
labels, and count the groups with value_counts().sort_index(). How many listings have no
group?
On purpose: First run the cut with bins=[1, 6, 29, 500] and count the missing values with
isna().sum(). Why are so many missing? Then fix it.

Answer: [1, 6, 29, 500] - is not inclusive for 1, fix: [0, 1, 6, 29, 500]

In [82]:
listings['stay'] = pd.cut(
    listings['minimum_nights'], 
    bins=[0, 1, 6, 29, 500], 
    labels=['1', '2-6', '7-29', '30-500']
)

listings['stay'].value_counts().sort_index()

listings['stay'].isna().sum() # 4

# listings['stay'] = pd.qcut(
#     listings['minimum_nights'], 
#     5, 
#     duplicates='drop'
# )

# listings['stay'].value_counts().sort_index()

np.int64(4)